# Review workbench — dual-model + diff  (Phase 1)

Two models check each transcript: **Model A** extracts (with citation validation), then a
**different Model B** reviews A's work for missed/wrong symptoms. You see the difference as a
highlighted transcript. Human adjudication + saving to a gold dataset come in Phase 2.

Spec: `specs/review_and_gold.md`. Sections 2 & 3 each cost one API call (different model each).

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

from clinical import (
    get_transcript,
    DEFAULT_MODEL,
    REVIEWER_MODEL,
    REVIEW_SYSTEM_PROMPT,
    build_review_user_message,
    get_reviewer_model,
    show_review_diff,
    show_symptom_evidence_matrix,
)
from graphs.validation_loop import build_graph

ENCOUNTER_DATE = "27.11.2025"
EXTRACTOR = build_graph()        # Model A: validation-loop extractor
REVIEWER = get_reviewer_model()  # Model B: independent reviewer

print(f"extractor (A): {DEFAULT_MODEL}")
print(f"reviewer  (B): {REVIEWER_MODEL}")

## 1. Pick a transcript

In [ ]:
TID = "transcript_1"
transcript = get_transcript(TID)
print(transcript)

## 2. Model A extracts  ⚠️ API call

In [ ]:
result = EXTRACTOR.invoke({"transcript": transcript, "encounter_date": ENCOUNTER_DATE})
extraction = result["extraction"]
print(f"attempts={result['attempts']}  citation errors={len(result['errors'])}  "
      f"problems={len(extraction.problems)}")

## 3. Model B reviews A's extraction  ⚠️ API call (different model)

B reads the transcript + A's output and proposes what's **missing / wrong / misattributed** —
each backed by a verbatim excerpt.

In [ ]:
review_messages = [
    SystemMessage(content=REVIEW_SYSTEM_PROMPT),
    HumanMessage(content=build_review_user_message(
        transcript, extraction.model_dump_json(indent=2), ENCOUNTER_DATE)),
]
critique = REVIEWER.invoke(review_messages)
print(f"reviewer proposed {len(critique.changes)} change(s)")
print("overall:", critique.overall)

## 4. The diff — 🟩 captured by A · 🟧 reviewer says missed

The transcript, highlighted. Green = the extractor cited it; orange = the reviewer thinks it
was missed. Below: the reviewer's concrete proposals.

In [ ]:
show_review_diff(transcript, extraction, critique)

## 5. The structured facts (A's extraction)

In [ ]:
show_symptom_evidence_matrix(extraction)

## Next — Phase 2

Adjudicate the reviewer's proposals (accept / reject), merge the accepted ones, and save the
signed-off result to `gold/<transcript_id>.json` with provenance (both models + your decisions).
See `specs/review_and_gold.md`.